# Ejercicio 5: Espacio Vectorial

## Objetivo de la práctica
- Implementar un Sistema de Recuperación de Información completo, desde la lectura del corpus hasta la recuperación de resultados.

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus

In [7]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

e:\Data\Universidad\6-7 Semestre\Recuperacion De La Informacion 26a\ir26a\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import pandas as pd

# Importamos el archivo
file_path = "wikipedia_text_corpus.csv"

# Cargamos el corpus
df = pd.read_csv(file_path)

# Verificamos
df.head()

,Unnamed: 0,text
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,2,Battery indicator\n\nA battery indicator (also...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...


2. Realiza las etapas de preprocesamiento sobre el corpus

2.2 Conversión a minúsculas

In [ ]:
df['text'] = df['text'].str.lower()

2.3 Eliminacion de caracteres especiales

In [20]:
import re

def limpiar_texto(texto):
    texto = re.sub(r'http\S+', '', texto)       # elimina URLs
    texto = re.sub(r'<.*?>', '', texto)          # elimina etiquetas HTML
    texto = re.sub(r'[^a-zA-Z\s]', '', texto)   # elimina no-letras
    texto = texto.strip()
    return texto

df['text'] = df['text'].apply(limpiar_texto)

2.3 Tokenización

In [ ]:
import nltk
import os

# Descargar específicamente 'punkt_tab' 
nltk.download('punkt')
nltk.download('punkt_tab') 

from nltk.tokenize import word_tokenize

# Aplicar tokenización
df['text'] = df['text'].fillna('')
df['tokens'] = df['text'].apply(word_tokenize)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\AdminPC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\AdminPC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


2.4 Eliminación de stopwords

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
# Creamos un conjunto de stopwords para una búsqueda rápida
stop_words = set(stopwords.words('english'))
# Definimos una función para eliminar stopwords
def eliminar_stopwords(texto):
    if not isinstance(texto, str):
        return ''
    tokens = word_tokenize(texto.lower())
    tokens_filtrados = [t for t in tokens if t.isalpha() and t not in stop_words]
    return ' '.join(tokens_filtrados)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\AdminPC\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [28]:
df['text_clean'] = df['text'].apply(eliminar_stopwords)


2.5 Stemming

In [ ]:
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

stemmer = PorterStemmer()
# Definir una función para aplicar stemming a un documento
def apply_stemming(doc):
    tokens = word_tokenize(doc)
    stemmed_tokens = [stemmer.stem(word) for word in tokens]
    return ' '.join(stemmed_tokens)

In [30]:
df['processed_review'] = df['text_clean'].apply(apply_stemming)

## Parte 1: Recuperación con TF-IDF

### Actividad:
3. Obtén la representación vectorial de los documentos utilizando el modelo TF-IDF

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Crear el vectorizador
vectorizer = TfidfVectorizer(max_features=10000)  # top 10k términos

# Ajustar y transformar el corpus preprocesado
tfidf_matrix = vectorizer.fit_transform(df['processed_review'])

print(f"Dimensiones de la matriz TF-IDF: {tfidf_matrix.shape}")
# (número de documentos, número de términos)

Dimensiones de la matriz TF-IDF: (10859, 10000)


4. A partir de un conjunto de 10 queries, verifica la recuperación del sistema

In [33]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 10 queries de ejemplo (puedes cambiarlas)
queries = [
    "artificial intelligence machine learning",
    "history of the roman empire",
    "climate change global warming",
    "football world cup championship",
    "space exploration moon mars",
    "python programming language",
    "world war two europe",
    "music theory composition",
    "human anatomy biology",
    "economics financial markets"
]

def recuperar_documentos(query, top_n=5):
    # Preprocesar la query igual que el corpus
    query_clean = eliminar_stopwords(query)
    query_stem  = apply_stemming(query_clean)
    
    # Vectorizar la query
    query_vec = vectorizer.transform([query_stem])
    
    # Calcular similitud coseno con todos los documentos
    similitudes = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # Obtener los top_n índices más similares
    top_indices = np.argsort(similitudes)[::-1][:top_n]
    
    print(f"\n Query: '{query}'")
    print("-" * 60)
    for i, idx in enumerate(top_indices):
        score = similitudes[idx]
        texto_preview = df['text'].iloc[idx][:120].replace('\n', ' ')
        print(f"  {i+1}. [score: {score:.4f}] {texto_preview}...")

# Ejecutar las 10 queries
for q in queries:
    recuperar_documentos(q, top_n=5)


 Query: 'artificial intelligence machine learning'
------------------------------------------------------------
  1. [score: 0.6363] outline of machine learning  the following outline is provided as an overview of and topical guide to machine learning m...
  2. [score: 0.4822] artificial psychology  artificial psychology is a theoretical discipline proposed by dan curtis b  the theory considers ...
  3. [score: 0.4256] docebolms  docebo is a software as a service artificial intelligence platform for elearning also known as a learning man...
  4. [score: 0.4183] teaching machine  teaching machines were originally mechanical devices they presented educational materials and taught s...
  5. [score: 0.3803] digital learning  digital learning is any type of learning that is accompanied by technology or by instructional practic...

 Query: 'history of the roman empire'
------------------------------------------------------------
  1. [score: 0.4754] list of byzantine inventions  this is a l

## Parte 2: Recuperación con BM25

### Actividad:
5. Implementa un sistema de recuperación usando el modelo BM25.

In [34]:
!pip install rank_bm25

  Using cached rank_bm25-0.2.2-py3-none-any.whl.metadata (3.2 kB)
Using cached rank_bm25-0.2.2-py3-none-any.whl (8.6 kB)



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
from rank_bm25 import BM25Okapi

# Por eso separamos cada documento en palabras individuales
corpus_tokenizado = [doc.split() for doc in df['processed_review']]

# Crear el modelo BM25 con el corpus tokenizado
# BM25Okapi es la variante más común y estable del algoritmo
# k1 y b son los hiperparámetros clásicos de BM25:
#   k1 (default 1.5): controla la saturación de frecuencia de términos
#   b  (default 0.75): controla la normalización por longitud del documento
bm25 = BM25Okapi(corpus_tokenizado, k1=1.5, b=0.75)

print(f"Modelo BM25 construido sobre {len(corpus_tokenizado)} documentos")
print(f"Ejemplo de tokens del doc 0: {corpus_tokenizado[0][:10]}...")

Modelo BM25 construido sobre 10859 documentos
Ejemplo de tokens del doc 0: ['anovo', 'anovo', 'formerli', 'novo', 'comput', 'servic', 'compani', 'base', 'beauvai', 'franc']...


6. Para el mismo conjunto de 10 queries, verifica la recuperación del sistema

In [36]:
def recuperar_bm25(query, top_n=5):
    """
    Recupera los documentos más relevantes para una query usando BM25.
    
    Parámetros:
        query  : texto de búsqueda en lenguaje natural
        top_n  : cantidad de documentos a retornar (default 5)
    """
    
    # Paso 1: aplicar el mismo preprocesamiento que usamos en el corpus
    # Es importante que la query pase por los mismos pasos (stopwords + stemming)
    # para que los términos sean comparables
    query_clean = eliminar_stopwords(query)
    query_stem  = apply_stemming(query_clean)
    
    # Paso 2: tokenizar la query (BM25 trabaja con listas de tokens)
    query_tokens = query_stem.split()
    
    # Paso 3: calcular el score BM25 de cada documento respecto a la query
    # get_scores() devuelve un array con un score por cada documento del corpus
    scores = bm25.get_scores(query_tokens)
    
    # Paso 4: ordenar los scores de mayor a menor y tomar los top_n
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_n]
    
    # Paso 5: mostrar los resultados
    print(f"\n Query: '{query}'")
    print("-" * 60)
    for i, idx in enumerate(top_indices):
        score        = scores[idx]
        # Mostramos los primeros 120 caracteres del texto original (sin preprocesar)
        texto_preview = df['text'].iloc[idx][:120].replace('\n', ' ')
        print(f"  {i+1}. [BM25 score: {score:.4f}] {texto_preview}...")


# --- Ejecutar las mismas 10 queries que usamos en TF-IDF ---
# Usamos las mismas para poder comparar resultados entre ambos modelos
for q in queries:
    recuperar_bm25(q, top_n=5)


 Query: 'artificial intelligence machine learning'
------------------------------------------------------------
  1. [BM25 score: 21.1608] outline of machine learning  the following outline is provided as an overview of and topical guide to machine learning m...
  2. [BM25 score: 19.0463] trace  trace inc is an irvine cabased information technology it company and managed service provider msp the company pro...
  3. [BM25 score: 18.7700] docebolms  docebo is a software as a service artificial intelligence platform for elearning also known as a learning man...
  4. [BM25 score: 18.7596] insilico medicine  insilico medicine is an american biotechnology company based in rockville in johns hopkins university...
  5. [BM25 score: 18.6678] cybernetics and systems  cybernetics and systems is a peerreviewed scientific journal of cybernetics and systems science...

 Query: 'history of the roman empire'
------------------------------------------------------------
  1. [BM25 score: 18.1630] engin

## Parte 3: Comparación de resultados

### Actividad:
7. Verifica cuáles documentos son recuperados (y en qué orden) por cada modelo de recuperación 

In [40]:
def comparar_modelos(query, top_n=5):
    # Preprocesar query
    query_clean  = eliminar_stopwords(query)
    query_stem   = apply_stemming(query_clean)
    query_tokens = query_stem.split()

    # TF-IDF
    query_vec  = vectorizer.transform([query_stem])
    sims_tfidf = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_tfidf  = np.argsort(sims_tfidf)[::-1][:top_n]

    # BM25
    scores_bm25 = bm25.get_scores(query_tokens)
    top_bm25    = sorted(range(len(scores_bm25)),
                         key=lambda i: scores_bm25[i], reverse=True)[:top_n]

    # Tabla
    print(f"\nQuery: '{query}'")
    print(f"{'Rank':<6} {'TF-IDF score | Documento':<40} {'BM25 score | Documento':<40}")
    print("-" * 86)

    for rank in range(top_n):
        idx_tfidf   = top_tfidf[rank]
        idx_bm25    = top_bm25[rank]
        col_tfidf   = f"{sims_tfidf[idx_tfidf]:.4f} | {df['text'].iloc[idx_tfidf][:25].replace(chr(10), ' ')}..."
        col_bm25    = f"{scores_bm25[idx_bm25]:.4f} | {df['text'].iloc[idx_bm25][:25].replace(chr(10), ' ')}..."
        print(f"  #{rank+1}   {col_tfidf:<40} {col_bm25:<40}")

    # Solapamiento
    coinciden    = set(top_tfidf) & set(top_bm25)
    solapamiento = len(coinciden) / top_n * 100
    print(f"\n  Documentos en comun: {len(coinciden)}/{top_n} ({solapamiento:.0f}%)")


# Ejecutar para las 10 queries
for q in queries:
    comparar_modelos(q, top_n=5)


Query: 'artificial intelligence machine learning'
Rank   TF-IDF score | Documento                 BM25 score | Documento                  
--------------------------------------------------------------------------------------
  #1   0.6363 | outline of machine learni...    21.1608 | outline of machine learni...  
  #2   0.4822 | artificial psychology  ar...    19.0463 | trace  trace inc is an ir...  
  #3   0.4256 | docebolms  docebo is a so...    18.7700 | docebolms  docebo is a so...  
  #4   0.4183 | teaching machine  teachin...    18.7596 | insilico medicine  insili...  
  #5   0.3803 | digital learning  digital...    18.6678 | cybernetics and systems  ...  

  Documentos en comun: 2/5 (40%)

Query: 'history of the roman empire'
Rank   TF-IDF score | Documento                 BM25 score | Documento                  
--------------------------------------------------------------------------------------
  #1   0.4754 | list of byzantine inventi...    18.1630 | engineering an empire 

In [41]:
# Conclusiones finales
solapamientos = []

for q in queries:
    query_clean  = eliminar_stopwords(q)
    query_stem   = apply_stemming(query_clean)
    query_tokens = query_stem.split()

    sims_tfidf  = cosine_similarity(vectorizer.transform([query_stem]), tfidf_matrix).flatten()
    scores_bm25 = bm25.get_scores(query_tokens)

    top_tfidf = set(np.argsort(sims_tfidf)[::-1][:5])
    top_bm25  = set(sorted(range(len(scores_bm25)),
                           key=lambda i: scores_bm25[i], reverse=True)[:5])

    solapamiento = len(top_tfidf & top_bm25) / 5 * 100
    solapamientos.append(solapamiento)

promedio = sum(solapamientos) / len(solapamientos)

print("=" * 50)
print("CONCLUSIONES")
print("=" * 50)
print(f"\nSolapamiento promedio entre modelos: {promedio:.1f}%\n")

for i, q in enumerate(queries):
    print(f"  Query {i+1}: {solapamientos[i]:.0f}% - '{q}'")

print("\n" + "-" * 50)

if promedio >= 70:
    print("Ambos modelos recuperan documentos similares.")
    print("TF-IDF es suficiente para este corpus.")
elif promedio >= 40:
    print("Los modelos difieren en varias queries.")
    print("BM25 es preferible por normalizar longitud de documentos.")
else:
    print("Los modelos recuperan documentos muy distintos.")
    print("BM25 es el modelo recomendado para este corpus.")

print(f"""
Razon: Wikipedia contiene articulos largos y variados.
BM25 penaliza documentos muy extensos y evita que
terminos frecuentes dominen el ranking, lo que lo
hace mas robusto que TF-IDF en este tipo de corpus.
""")

CONCLUSIONES

Solapamiento promedio entre modelos: 34.0%

  Query 1: 40% - 'artificial intelligence machine learning'
  Query 2: 60% - 'history of the roman empire'
  Query 3: 20% - 'climate change global warming'
  Query 4: 20% - 'football world cup championship'
  Query 5: 40% - 'space exploration moon mars'
  Query 6: 80% - 'python programming language'
  Query 7: 20% - 'world war two europe'
  Query 8: 20% - 'music theory composition'
  Query 9: 0% - 'human anatomy biology'
  Query 10: 40% - 'economics financial markets'

--------------------------------------------------
Los modelos recuperan documentos muy distintos.
BM25 es el modelo recomendado para este corpus.

Razon: Wikipedia contiene articulos largos y variados.
BM25 penaliza documentos muy extensos y evita que
terminos frecuentes dominen el ranking, lo que lo
hace mas robusto que TF-IDF en este tipo de corpus.

